In [1]:
import pandas as pd 

In [2]:
df = pd.read_csv('products.csv').reset_index().rename(columns={"index": "id"})

In [3]:
df.head()

,id,product,description
0,0,MacBook Air,A lightweight Apple laptop suitable for work s...
1,1,iPhone 14,Apple's smartphone for communication internet ...
2,2,Samsung Galaxy S24+,Samsung's flagship smartphone with advanced fe...
3,3,iPad Pro,Apple's touchscreen tablet ideal for browsing ...
4,4,Apple Watch Series 9,A wearable device that tracks fitness and disp...


In [4]:
df.iloc[0].description

'A lightweight Apple laptop suitable for work study and entertainment.'

# Semantic Search

Aims to understand the meaning and context of search queries based on semantic understanding. Not just keyword searches.

Converts both the query and the documents to numerical embeddings.

emb(query) should be approximately equal to the emb(document) 

In [5]:
def preprocess_text(text: str) -> str:
    """ Convert text to lowercase and remove extra spaces"""
    return text.lower().strip()

In [6]:
df['description'] = df.description.apply(preprocess_text)

In [7]:
df.iloc[0].description

'a lightweight apple laptop suitable for work study and entertainment.'

## Convert to embeddings

In [8]:
import os
import dotenv
dotenv.load_dotenv(dotenv_path='.env')

from openai import OpenAI

In [9]:
client = OpenAI(
  api_key=os.environ['OPENAI_API_KEY'],  # this is also the default, it can be omitted
)

In [34]:
def generate_embeddings(texts: list[str], oai_client: OpenAI, model: str = "text-embedding-ada-002"):
    response = oai_client.embeddings.create(input = texts, model = model)
    return [e.embedding for e in response.data]

In [35]:
embeddings = generate_embeddings(texts=df.description.tolist(), oai_client=client)

In [38]:
len(embeddings[0])

1536

# Vector Store!

Time to use these embeddings in a vector store using some kind of index

In [26]:
import faiss
import numpy as np

In [30]:
dimension = len(embeddings[0].embedding)
dimension

1536

In [31]:
index = faiss.IndexFlatL2(dimension)
index


<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7f03babb9bc0> >

In [33]:
def add_to_vector_store(emb, ids):
    """Add embeddings and their corresponding IDs to FAISS"""
    index.add(np.array(emb))

In [39]:
add_to_vector_store(emb=embeddings, ids=df.id.tolist())

In [40]:
index

<faiss.swigfaiss_avx2.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x7f03babb9bc0> >

## Querying

- convert query to an embedding
- find the closest matches

In [75]:
def search(query: str, oai_client: OpenAI, num_nearest_neighbours: int = 10) -> tuple[list[int], list[float]]:
    """Search for relevant documents based on a query"""
    query_embedding = generate_embeddings([query], oai_client=oai_client)[0]
    distances, indices = index.search(np.array([query_embedding]), k = num_nearest_neighbours)
    return distances, indices

In [ ]:
matched_docs = df.iloc[I[0]]

In [76]:
query = "a wearable device"

In [77]:
distances, indices = search(query=query, oai_client=client)

In [80]:
df.iloc[indices[0]]

,id,product,description
4,4,Apple Watch Series 9,a wearable device that tracks fitness and disp...
68,68,Garmin Venu 3,a smartwatch with advanced health monitoring.
64,64,Fitbit Versa 4,a smartwatch for fitness and health tracking.
151,151,Samsung Galaxy Watch6,a smartwatch with advanced health features.
14,14,Fitbit Charge 6,a fitness tracker for monitoring health metric...
130,130,Samsung Galaxy Fit3,a fitness band for health tracking.
60,60,Samsung Galaxy Watch6 Classic,a stylish smartwatch with health tracking feat...
155,155,Fitbit Luxe,a stylish fitness tracker with color display.
146,146,Tile Sticker (2024),a small bluetooth tracker for gadgets.
101,101,Samsung Galaxy Watch5 Pro,a durable smartwatch for outdoor activities.
